# Data Augmentation Validation Metrics

This notebook computes validation metrics for augmented datasets:
- **FID (Fréchet Inception Distance)**: Measures distribution distance between real and generated images
- **DINO Structural Similarity**: Measures structural similarity using DINO self-supervised features

## Dataset Structure
```
augmentation_data/
├── construction_site-test/        # Original data
├── night/construction_site-test/  # Night augmentation
├── small/construction_site-test/  # Small object augmentation
└── weather/construction_site_test/ # Weather augmentation (rain, snow, etc.)
```

In [1]:
import os
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


## Configuration

In [2]:
# Base paths
BASE_DIR = Path("/users/PGS0407/binben14/VietHuy/construction-site")
AUG_DATA_DIR = BASE_DIR / "augmentation_data"

# Dataset paths
DATASETS = {
    "original": AUG_DATA_DIR / "construction_site-test" / "images",
    "night": AUG_DATA_DIR / "night" / "construction_site-test" / "images",
    "small": AUG_DATA_DIR / "small" / "construction_site-test" / "images",
}

# Weather augmentations (multiple styles)
weather_base = AUG_DATA_DIR / "weather" / "construction_site_test" / "day" / "filtered"
if weather_base.exists():
    for style_dir in sorted(weather_base.iterdir()):
        if style_dir.is_dir() and (style_dir / "images").exists():
            DATASETS[f"weather_{style_dir.name}"] = style_dir / "images"

# Display available datasets
print("Available datasets:")
for name, path in DATASETS.items():
    if path.exists():
        n_images = len(list(path.glob("*.jpg"))) + len(list(path.glob("*.png")))
        print(f"  {name}: {path} ({n_images} images)")
    else:
        print(f"  {name}: {path} (NOT FOUND)")

Available datasets:
  original: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/construction_site-test/images (3004 images)
  night: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/night/construction_site-test/images (3004 images)
  small: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/small/construction_site-test/images (1323 images)
  weather_style_rain_0: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/weather/construction_site_test/day/filtered/style_rain_0/images (2211 images)
  weather_style_rain_1: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/weather/construction_site_test/day/filtered/style_rain_1/images (1243 images)
  weather_style_rain_2: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/weather/construction_site_test/day/filtered/style_rain_2/images (1208 images)
  weather_style_snow_0: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/

## 1. FID (Fréchet Inception Distance)

FID measures the distance between feature distributions of real and generated images using InceptionV3.

Lower FID = Better quality and more similar distribution

In [3]:
from torchvision import transforms
from torchvision.models import inception_v3, Inception_V3_Weights
from scipy import linalg

class FIDCalculator:
    def __init__(self, device='cuda'):
        self.device = device
        # Load InceptionV3 pretrained model
        self.model = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
        self.model.fc = torch.nn.Identity()  # Remove final classification layer
        self.model = self.model.to(device)
        self.model.eval()
        
        # Transform for InceptionV3 (299x299)
        self.transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    
    def get_features(self, image_paths: List[Path], batch_size: int = 32) -> np.ndarray:
        """Extract InceptionV3 features for a list of images."""
        features = []
        
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting features"):
            batch_paths = image_paths[i:i+batch_size]
            batch_images = []
            
            for path in batch_paths:
                try:
                    img = Image.open(path).convert('RGB')
                    img_tensor = self.transform(img)
                    batch_images.append(img_tensor)
                except Exception as e:
                    print(f"Error loading {path}: {e}")
                    continue
            
            if len(batch_images) == 0:
                continue
                
            batch_tensor = torch.stack(batch_images).to(self.device)
            
            with torch.no_grad():
                feat = self.model(batch_tensor)
                features.append(feat.cpu().numpy())
        
        return np.concatenate(features, axis=0)
    
    def calculate_statistics(self, features: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Calculate mean and covariance of features."""
        mu = np.mean(features, axis=0)
        sigma = np.cov(features, rowvar=False)
        return mu, sigma
    
    def calculate_fid(self, mu1, sigma1, mu2, sigma2, eps=1e-6) -> float:
        """Calculate FID between two distributions."""
        diff = mu1 - mu2
        
        # Product might be almost singular
        covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
        
        if not np.isfinite(covmean).all():
            offset = np.eye(sigma1.shape[0]) * eps
            covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
        
        # Numerical error might give slight imaginary component
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        
        tr_covmean = np.trace(covmean)
        
        fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean
        return float(fid)
    
    def compute_fid_between_folders(self, real_path: Path, gen_path: Path, 
                                     max_samples: Optional[int] = None) -> float:
        """Compute FID between two image folders."""
        # Get image paths
        real_images = sorted(list(real_path.glob("*.jpg")) + list(real_path.glob("*.png")))
        gen_images = sorted(list(gen_path.glob("*.jpg")) + list(gen_path.glob("*.png")))
        
        if max_samples:
            real_images = real_images[:max_samples]
            gen_images = gen_images[:max_samples]
        
        print(f"Real images: {len(real_images)}, Generated images: {len(gen_images)}")
        
        # Extract features
        print("\nExtracting real image features...")
        real_features = self.get_features(real_images)
        
        print("\nExtracting generated image features...")
        gen_features = self.get_features(gen_images)
        
        # Calculate statistics
        mu1, sigma1 = self.calculate_statistics(real_features)
        mu2, sigma2 = self.calculate_statistics(gen_features)
        
        # Calculate FID
        fid = self.calculate_fid(mu1, sigma1, mu2, sigma2)
        return fid

print("FIDCalculator class defined.")

FIDCalculator class defined.


## 2. DINO Structural Similarity

DINO (self-DIstillation with NO labels) extracts semantically meaningful features.
We use DINO features to measure structural similarity between original and augmented images.

In [4]:
class DINOStructuralSimilarity:
    def __init__(self, device='cuda', model_name='dinov2_vits14'):
        self.device = device
        
        # Load DINOv2 model from torch hub
        print(f"Loading {model_name}...")
        self.model = torch.hub.load('facebookresearch/dinov2', model_name)
        self.model = self.model.to(device)
        self.model.eval()
        
        # Transform for DINO (224x224 or 518x518 for large)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        print("DINO model loaded.")
    
    def get_features(self, image_paths: List[Path], batch_size: int = 32) -> np.ndarray:
        """Extract DINO features for a list of images."""
        features = []
        
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting DINO features"):
            batch_paths = image_paths[i:i+batch_size]
            batch_images = []
            
            for path in batch_paths:
                try:
                    img = Image.open(path).convert('RGB')
                    img_tensor = self.transform(img)
                    batch_images.append(img_tensor)
                except Exception as e:
                    continue
            
            if len(batch_images) == 0:
                continue
                
            batch_tensor = torch.stack(batch_images).to(self.device)
            
            with torch.no_grad():
                feat = self.model(batch_tensor)
                features.append(feat.cpu().numpy())
        
        return np.concatenate(features, axis=0)
    
    def cosine_similarity(self, feat1: np.ndarray, feat2: np.ndarray) -> float:
        """Calculate mean cosine similarity between feature sets."""
        # Normalize features
        feat1_norm = feat1 / (np.linalg.norm(feat1, axis=1, keepdims=True) + 1e-8)
        feat2_norm = feat2 / (np.linalg.norm(feat2, axis=1, keepdims=True) + 1e-8)
        
        # Calculate pairwise similarities for matched images
        min_len = min(len(feat1), len(feat2))
        similarities = np.sum(feat1_norm[:min_len] * feat2_norm[:min_len], axis=1)
        
        return float(np.mean(similarities))
    
    def compute_structural_similarity(self, real_path: Path, gen_path: Path,
                                       match_by_name: bool = True,
                                       max_samples: Optional[int] = None) -> dict:
        """Compute DINO structural similarity between two folders."""
        
        if match_by_name:
            # Match images by filename
            real_images = {p.stem: p for p in real_path.glob("*.jpg")}
            real_images.update({p.stem: p for p in real_path.glob("*.png")})
            
            gen_images = {p.stem: p for p in gen_path.glob("*.jpg")}
            gen_images.update({p.stem: p for p in gen_path.glob("*.png")})
            
            # Find common images
            common_ids = sorted(set(real_images.keys()) & set(gen_images.keys()))
            
            if max_samples:
                common_ids = common_ids[:max_samples]
            
            real_paths = [real_images[id] for id in common_ids]
            gen_paths = [gen_images[id] for id in common_ids]
            
            print(f"Matched {len(common_ids)} image pairs by filename")
        else:
            real_paths = sorted(list(real_path.glob("*.jpg")) + list(real_path.glob("*.png")))
            gen_paths = sorted(list(gen_path.glob("*.jpg")) + list(gen_path.glob("*.png")))
            
            if max_samples:
                real_paths = real_paths[:max_samples]
                gen_paths = gen_paths[:max_samples]
        
        print(f"Real images: {len(real_paths)}, Generated images: {len(gen_paths)}")
        
        # Extract features
        print("\nExtracting real image DINO features...")
        real_features = self.get_features(real_paths)
        
        print("\nExtracting generated image DINO features...")
        gen_features = self.get_features(gen_paths)
        
        # Calculate metrics
        cosine_sim = self.cosine_similarity(real_features, gen_features)
        
        # Calculate per-image similarities
        min_len = min(len(real_features), len(gen_features))
        real_norm = real_features[:min_len] / (np.linalg.norm(real_features[:min_len], axis=1, keepdims=True) + 1e-8)
        gen_norm = gen_features[:min_len] / (np.linalg.norm(gen_features[:min_len], axis=1, keepdims=True) + 1e-8)
        per_image_sim = np.sum(real_norm * gen_norm, axis=1)
        
        return {
            'mean_cosine_similarity': cosine_sim,
            'std_cosine_similarity': float(np.std(per_image_sim)),
            'min_cosine_similarity': float(np.min(per_image_sim)),
            'max_cosine_similarity': float(np.max(per_image_sim)),
            'n_pairs': min_len,
            'per_image_similarities': per_image_sim
        }

print("DINOStructuralSimilarity class defined.")

DINOStructuralSimilarity class defined.


## 3. Initialize Calculators

In [5]:
# Initialize FID calculator
print("Initializing FID Calculator...")
fid_calc = FIDCalculator(device=device)
print("FID Calculator ready.\n")

# Initialize DINO calculator
print("Initializing DINO Structural Similarity...")
dino_calc = DINOStructuralSimilarity(device=device)
print("DINO Calculator ready.")

Initializing FID Calculator...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /users/PGS0407/binben14/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 247MB/s] 


FID Calculator ready.

Initializing DINO Structural Similarity...
Loading dinov2_vits14...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /users/PGS0407/binben14/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /users/PGS0407/binben14/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 228MB/s] 


DINO model loaded.
DINO Calculator ready.


## 4. Compute Metrics for All Datasets

In [6]:
# Settings
MAX_SAMPLES = None  # Set to a number to limit samples, e.g., 500
REFERENCE_DATASET = "original"

# Datasets to evaluate (all except reference)
datasets_to_eval = {k: v for k, v in DATASETS.items() if k != REFERENCE_DATASET and v.exists()}

print(f"Reference dataset: {REFERENCE_DATASET}")
print(f"Datasets to evaluate: {list(datasets_to_eval.keys())}")

Reference dataset: original
Datasets to evaluate: ['night', 'small', 'weather_style_rain_0', 'weather_style_rain_1', 'weather_style_rain_2', 'weather_style_snow_0', 'weather_style_snow_1', 'weather_style_snow_2']


In [ ]:
# Compute metrics
results = []

reference_path = DATASETS[REFERENCE_DATASET]

for name, path in datasets_to_eval.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"Path: {path}")
    print('='*60)
    
    try:
        # Compute FID
        print("\n--- Computing FID ---")
        fid_score = fid_calc.compute_fid_between_folders(
            reference_path, path, max_samples=MAX_SAMPLES
        )
        print(f"FID Score: {fid_score:.4f}")
        
        # Compute DINO structural similarity
        print("\n--- Computing DINO Structural Similarity ---")
        dino_results = dino_calc.compute_structural_similarity(
            reference_path, path, match_by_name=True, max_samples=MAX_SAMPLES
        )
        print(f"DINO Mean Cosine Similarity: {dino_results['mean_cosine_similarity']:.4f}")
        print(f"DINO Std: {dino_results['std_cosine_similarity']:.4f}")
        
        results.append({
            'dataset': name,
            'fid_score': fid_score,
            'dino_mean_sim': dino_results['mean_cosine_similarity'],
            'dino_std_sim': dino_results['std_cosine_similarity'],
            'dino_min_sim': dino_results['min_cosine_similarity'],
            'dino_max_sim': dino_results['max_cosine_similarity'],
            'n_pairs': dino_results['n_pairs']
        })
        
    except Exception as e:
        print(f"Error evaluating {name}: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*60)
print("Evaluation Complete!")
print("="*60)


Evaluating: night
Path: /users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/night/construction_site-test/images

--- Computing FID ---
Real images: 3004, Generated images: 3004

Extracting real image features...


Extracting features:  21%|██▏       | 20/94 [03:29<13:41, 11.10s/it]

## 5. Results Summary

In [ ]:
# Create results DataFrame
df_results = pd.DataFrame(results)

# Sort by FID score (lower is better)
df_results = df_results.sort_values('fid_score')

print("\n" + "="*80)
print("VALIDATION METRICS SUMMARY")
print("="*80)
print(f"\nReference dataset: {REFERENCE_DATASET}")
print(f"Reference path: {DATASETS[REFERENCE_DATASET]}")
print("\n")

# Display results
display_cols = ['dataset', 'fid_score', 'dino_mean_sim', 'dino_std_sim', 'n_pairs']
print(df_results[display_cols].to_string(index=False))

# Save to CSV
output_csv = BASE_DIR / "validation_data" / "validation_results.csv"
df_results.to_csv(output_csv, index=False)
print(f"\nResults saved to: {output_csv}")

In [ ]:
# Visualization
if len(df_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # FID Score comparison
    ax1 = axes[0]
    colors = plt.cm.viridis(np.linspace(0, 1, len(df_results)))
    bars1 = ax1.barh(df_results['dataset'], df_results['fid_score'], color=colors)
    ax1.set_xlabel('FID Score (lower is better)')
    ax1.set_title('FID Score Comparison')
    ax1.invert_yaxis()
    for i, (bar, val) in enumerate(zip(bars1, df_results['fid_score'])):
        ax1.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', 
                va='center', fontsize=9)
    
    # DINO Similarity comparison
    ax2 = axes[1]
    df_sorted_dino = df_results.sort_values('dino_mean_sim', ascending=False)
    bars2 = ax2.barh(df_sorted_dino['dataset'], df_sorted_dino['dino_mean_sim'], 
                     xerr=df_sorted_dino['dino_std_sim'], color=colors, capsize=3)
    ax2.set_xlabel('DINO Cosine Similarity (higher is better)')
    ax2.set_title('DINO Structural Similarity Comparison')
    ax2.set_xlim(0, 1)
    ax2.invert_yaxis()
    for i, (bar, val) in enumerate(zip(bars2, df_sorted_dino['dino_mean_sim'])):
        ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}', 
                va='center', fontsize=9)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = BASE_DIR / "validation_data" / "validation_metrics_comparison.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Figure saved to: {fig_path}")
    
    plt.show()

## 6. Detailed Analysis: Per-Image DINO Similarity Distribution

In [ ]:
def analyze_single_dataset(name: str, path: Path, reference_path: Path, 
                           dino_calc: DINOStructuralSimilarity,
                           max_samples: int = 500):
    """Detailed analysis of a single dataset's DINO similarity."""
    
    print(f"Analyzing: {name}")
    results = dino_calc.compute_structural_similarity(
        reference_path, path, match_by_name=True, max_samples=max_samples
    )
    
    per_image_sim = results['per_image_similarities']
    
    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(per_image_sim, bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(results['mean_cosine_similarity'], color='red', 
                    linestyle='--', label=f"Mean: {results['mean_cosine_similarity']:.4f}")
    axes[0].set_xlabel('Cosine Similarity')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{name} - DINO Similarity Distribution')
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(per_image_sim, vert=True)
    axes[1].set_ylabel('Cosine Similarity')
    axes[1].set_title(f'{name} - Box Plot')
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"\nStatistics:")
    print(f"  Mean: {results['mean_cosine_similarity']:.4f}")
    print(f"  Std:  {results['std_cosine_similarity']:.4f}")
    print(f"  Min:  {results['min_cosine_similarity']:.4f}")
    print(f"  Max:  {results['max_cosine_similarity']:.4f}")
    print(f"  Median: {np.median(per_image_sim):.4f}")
    print(f"  25th percentile: {np.percentile(per_image_sim, 25):.4f}")
    print(f"  75th percentile: {np.percentile(per_image_sim, 75):.4f}")
    
    return per_image_sim

In [ ]:
# Analyze specific dataset (uncomment to run)
# analyze_single_dataset("night", DATASETS["night"], DATASETS["original"], dino_calc, max_samples=500)

## 7. Utility Functions

In [ ]:
def quick_validate(aug_path: str, ref_path: str = None, max_samples: int = 200):
    """
    Quick validation function for a single augmentation dataset.
    
    Args:
        aug_path: Path to augmented images folder
        ref_path: Path to reference images folder (default: original dataset)
        max_samples: Maximum number of samples to use
    
    Returns:
        dict: FID score and DINO similarity metrics
    """
    aug_path = Path(aug_path)
    ref_path = Path(ref_path) if ref_path else DATASETS["original"]
    
    print(f"Reference: {ref_path}")
    print(f"Augmented: {aug_path}")
    print(f"Max samples: {max_samples}\n")
    
    # FID
    print("Computing FID...")
    fid = fid_calc.compute_fid_between_folders(ref_path, aug_path, max_samples=max_samples)
    
    # DINO
    print("\nComputing DINO similarity...")
    dino = dino_calc.compute_structural_similarity(ref_path, aug_path, 
                                                    match_by_name=True, 
                                                    max_samples=max_samples)
    
    print(f"\n{'='*40}")
    print(f"RESULTS")
    print(f"{'='*40}")
    print(f"FID Score: {fid:.4f} (lower is better)")
    print(f"DINO Similarity: {dino['mean_cosine_similarity']:.4f} ± {dino['std_cosine_similarity']:.4f}")
    print(f"  (higher is better, 1.0 = identical)")
    
    return {
        'fid': fid,
        'dino_mean': dino['mean_cosine_similarity'],
        'dino_std': dino['std_cosine_similarity']
    }

print("quick_validate() function defined.")
print("\nUsage: quick_validate('/path/to/augmented/images')")

In [ ]:
# Example usage:
# quick_validate(DATASETS["night"])